In [1]:
# =============================================================================
#  TITANIC — PIPELINE DIDÁTICO DE CLASSIFICAÇÃO BINÁRIA
#  Dataset: Titanic (via seaborn / Kaggle clássico)
#
#  Estrutura:
#    PARTE 1  — Configurações e importações
#    PARTE 2  — Carregamento do dataset
#    PARTE 3  — EDA completa (estatísticas, visualizações, balanceamento) Análise Exploratória de Dados
#    PARTE 4  — Pré-processamento e engenharia de features
#    PARTE 5  — Divisão treino / teste
#    PARTE 6  — Modelos tradicionais (6 modelos)
#    PARTE 7  — Modelos deep learning com PyTorch (3 arquiteturas)
#    PARTE 8  — Comparação e escolha do melhor modelo
#    PARTE 9  — Serialização para produção
# =============================================================================

In [1]:
!pip install torch


[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: C:\Users\calab\.venv310\Scripts\python.exe -m pip install --upgrade pip


In [2]:
# ─────────────────────────────────────────────────────────────────────────────
# PARTE 1 — CONFIGURAÇÕES E IMPORTAÇÕES (reorganizado + comentários)
# ─────────────────────────────────────────────────────────────────────────────

# =========================
# 1) BIBLIOTECA PADRÃO (stdlib)
# =========================
import warnings  # controla/exibe/oculta avisos (warnings); útil p/ deixar output limpo
import os        # interação com o SO (paths, env vars, listar pastas); 
import json      # ler/escrever JSON (configs, logs, resultados)
import time      # medir tempo e fazer delays (benchmark, sleep); 
from pathlib import Path  # caminhos de arquivo “modernos” e portáveis; 
from datetime import datetime  # datas/horários (timestamps, logs, versionamento de arquivos)

# =========================
# 2) SERIALIZAÇÃO / UTILITÁRIOS DE TERCEIROS
# =========================
import joblib  # salvar/carregar objetos grandes (modelos sklearn, scalers); 

# =========================
# 3) COMPUTAÇÃO / DADOS
# =========================
import numpy as np  # arrays e álgebra numérica; base para ML/ETL
import pandas as pd  # DataFrames para manipular dados tabulares (CSV/Excel, limpeza, etc.)

# =========================
# 4) VISUALIZAÇÃO
# =========================
import matplotlib.pyplot as plt  # API principal de gráficos
import matplotlib.gridspec as gridspec  # layout avançado de subplots; 
import seaborn as sns  # gráficos estatísticos (heatmap, distplot, boxplot); 

# =========================
# 5) SCIKIT-LEARN (ML clássico + pré-processamento + métricas)
# =========================
from sklearn.model_selection import train_test_split, cross_val_score  # split treino/teste e validação cruzada; 
from sklearn.preprocessing import StandardScaler, LabelEncoder  # normalização e codificação de labels; 
from sklearn.metrics import (  # métricas de avaliação
    accuracy_score,          # acurácia
    f1_score,                # F1
    precision_score,         # precisão
    recall_score,            # recall
    roc_auc_score,           # AUC
    confusion_matrix,        # matriz de confusão (numérica)
    classification_report,   # relatório formatado
    roc_curve,               # pontos p/ curva ROC
    ConfusionMatrixDisplay,  # plot pronto da matriz; 
)

from sklearn.utils.class_weight import compute_class_weight  # calcula pesos p/ classes desbalanceadas; 
# ---- Modelos clássicos (use só os que você realmente treinar) ----
from sklearn.linear_model import LogisticRegression  # baseline forte p/ classificação linear; 
from sklearn.tree import DecisionTreeClassifier      # árvore simples; 
from sklearn.ensemble import (                       # ensembles
    RandomForestClassifier,      # bagging de árvores; ótimo baseline
    GradientBoostingClassifier,  # boosting “clássico”
    AdaBoostClassifier,          # boosting adaptativo; 
)
from sklearn.svm import SVC  # SVM; 
from sklearn.naive_bayes import GaussianNB  # Naive Bayes; 

# =========================
# 6) PYTORCH (Deep Learning)
# =========================
import torch  # tensores, GPU, autograd, etc.
import torch.nn as nn  # camadas/losses (Linear, ReLU, CrossEntropy, etc.)
import torch.optim as optim  # otimizadores (SGD/Adam/etc.)

from torch.utils.data import DataLoader, TensorDataset  # batching e datasets com tensores; 

# =========================
# 7) CONFIGURAÇÃO FINAL
# =========================
warnings.filterwarnings('ignore')  # oculta warnings; 

In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# DIRETÓRIOS DE SAÍDA DO PROJETO
# ─────────────────────────────────────────────────────────────────────────────

# Diretório raiz onde todos os artefatos do projeto serão salvos
# (resultados, gráficos, modelos, arquivos finais para produção)
OUT = Path('titanic_output')

# Subdiretório para gráficos gerados na EDA e na avaliação dos modelos
PLOTS = OUT / 'plots'

# Subdiretório para modelos treinados
# (ex.: .pkl do sklearn, checkpoints, pesos PyTorch)
MDIR = OUT / 'models'

# Subdiretório para artefatos finais de produção
# (modelo definitivo, scaler, config.json, código de inferência)
PROD = OUT / 'production'

# Cria todos os diretórios acima, se ainda não existirem
# parents=True   → cria diretórios intermediários
# exist_ok=True  → evita erro caso já existam
for d in [OUT, PLOTS, MDIR, PROD]:
    d.mkdir(parents=True, exist_ok=True)


# ─────────────────────────────────────────────────────────────────────────────
# CONSTANTES DO EXPERIMENTO
# ─────────────────────────────────────────────────────────────────────────────

# Semente global de aleatoriedade
# Garante reprodutibilidade: mesmos splits, mesmos pesos iniciais, mesmos batches
SEED = 42

# Percentual do dataset reservado para teste
# 0.20 → 80% treino / 20% teste (padrão para datasets pequenos como Titanic)
TEST_SIZE = 0.20

# Número de épocas de treinamento para modelos de Deep Learning
# Uma época = uma passagem completa pelo conjunto de treino
EPOCHS = 60

# Learning Rate (taxa de aprendizado)
# Controla o tamanho dos passos do otimizador no espaço de parâmetros
# 1e-3 é um valor padrão estável para Adam / AdamW
LR = 1e-3

# Tamanho do batch (mini-lote)
# Quantidade de amostras processadas antes de uma atualização de gradiente
# Titanic tem ~900 linhas → batch pequeno ajuda na generalização
BATCH_SIZE = 32


# ─────────────────────────────────────────────────────────────────────────────
# CONTROLE DE REPRODUTIBILIDADE
# ─────────────────────────────────────────────────────────────────────────────

# Fixa a aleatoriedade do NumPy
# Afeta: operações aleatórias, inicializações, embaralhamentos
np.random.seed(SEED)

# Fixa a aleatoriedade do PyTorch (CPU)
# Afeta: inicialização dos pesos, dropout, ordem dos batches
torch.manual_seed(SEED)


# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURAÇÃO DE DISPOSITIVO (CPU / GPU)
# ─────────────────────────────────────────────────────────────────────────────

# Define automaticamente onde os tensores e modelos PyTorch irão rodar
# cuda → GPU disponível
# cpu  → execução padrão caso não haja GPU
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Log informativo para garantir transparência do hardware usado
print(f"Dispositivo PyTorch: {DEVICE}")

# Se estiver usando GPU, exibe o nome da placa
# Útil para logs, benchmarks e reprodutibilidade
if DEVICE.type == 'cuda':
    print(f"  GPU: {torch.cuda.get_device_name(0)}")


Dispositivo PyTorch: cuda
  GPU: NVIDIA GeForce GTX 1660 Ti


In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# ESTILO GLOBAL DOS GRÁFICOS (Matplotlib)
# ─────────────────────────────────────────────────────────────────────────────

plt.rcParams.update({
    # Resolução da figura em DPI (dots per inch)
    # Valores maiores deixam o gráfico mais nítido, especialmente ao salvar PNG
    # 120 é um bom equilíbrio entre qualidade e tamanho do arquivo
    'figure.dpi': 120,

    # Tamanho padrão das figuras (largura, altura) em polegadas
    # (10, 5) gera gráficos horizontais, ideais para notebooks e relatórios
    'figure.figsize': (10, 5),

    # Remove a borda superior dos gráficos
    # Deixa o visual mais limpo e moderno (estilo "seaborn")
    'axes.spines.top': False,

    # Remove a borda direita dos gráficos
    # Evita poluição visual e melhora a leitura
    'axes.spines.right': False,

    # Tamanho padrão da fonte (labels, ticks, títulos simples)
    # 11 é confortável para leitura em tela e PDFs
    'font.size': 11,
})


# ─────────────────────────────────────────────────────────────────────────────
# PALETA DE CORES DO PROJETO
# ─────────────────────────────────────────────────────────────────────────────

# Paleta fixa usada em todos os gráficos de classificação
# Mantém consistência visual ao longo da EDA e da avaliação
PALETTE = [
    '#2196F3',  # azul — classe 0 (não sobreviveu)
    '#F44336',  # vermelho — classe 1 (sobreviveu)
]

In [5]:
# =============================================================================
# PARTE 2 — CARREGAMENTO DO DATASET
# =============================================================================

def load_data() -> pd.DataFrame:
    """
    Carrega o dataset Titanic diretamente via seaborn (fonte: GitHub/datasets).

    Por que o Titanic?
    ──────────────────
    • Clássico de sala de aula: amplamente documentado, resultados comparáveis.
    • Tamanho ideal para notebooks (~891 passageiros): rápido de treinar,
      grande o suficiente para demonstrar técnicas estatísticas.
    • Variáveis mistas: numéricas (Age, Fare), categóricas (Sex, Embarked),
      ordinais (Pclass), texto livre (Name) — exercita todo o pré-processamento.
    • Desbalanceamento REAL: ~38% sobreviveram, ~62% não — requer tratamento.
    • Narrativa intuitiva: qualquer pessoa entende o problema de negócio.

    Retorna:
        pd.DataFrame com 891 linhas e 15 colunas originais.
    """
    print("\n" + "="*70)
    print("PARTE 2 — CARREGAMENTO DO DATASET")
    print("="*70)

    df = sns.load_dataset('titanic')   # download automático via seaborn
    print(f"Dataset carregado:  {df.shape[0]} passageiros × {df.shape[1]} colunas")
    print(f"Colunas: {list(df.columns)}")
    return df
    

In [6]:
# =============================================================================
# PARTE 3 — ANÁLISE EXPLORATÓRIA DE DADOS (EDA)
# =============================================================================

def run_eda(df: pd.DataFrame) -> None:
    """
    EDA completa em 10 etapas, com justificativa de cada passo e interpretação
    dos resultados. Todos os gráficos são salvos em PLOTS/.

    Por que fazer EDA antes de modelar?
    ─────────────────────────────────────
    A EDA revela problemas (nulos, outliers, correlações espúrias) antes que
    eles contaminem o modelo. Skipping EDA é a causa mais comum de modelos
    que parecem bons no notebook mas falham em produção.
    """
    print("\n" + "="*70)
    print("PARTE 3 — ANÁLISE EXPLORATÓRIA DE DADOS (EDA)")
    print("="*70)

    # ──────────────────────────────────────────────────────────────────────────
    # EDA 3.1 — Visão geral: shape, tipos e nulos
    # ──────────────────────────────────────────────────────────────────────────
    print("\n── 3.1  Visão geral ──────────────────────────────────────────────────")
    print("""
POR QUE ESTA ETAPA?
  Antes de qualquer análise precisamos entender o que temos: quantas linhas,
  quais tipos de dado e se há valores ausentes. Colunas com muitos nulos
  precisam de estratégia especial (imputação ou descarte).
""")
    print(df.info())
    print(df.dtypes)
    print(f"\nShape: {df.shape}")
    print(f"\nValores nulos por coluna:")
    nulos = df.isnull().sum()
    nulos_pct = (nulos / len(df) * 100).round(1)
    resumo_nulos = pd.DataFrame({'nulos': nulos, '(%)': nulos_pct})
    print(resumo_nulos[resumo_nulos['nulos'] > 0].to_string())

    print("""
INTERPRETAÇÃO:
  • 'age'     : 177 nulos (19,9%) — imputação pela mediana por Pclass
  • 'deck'    : 688 nulos (77,2%) — coluna descartada (informação insuficiente)
  • 'embarked': 2 nulos  ( 0,2%) — preenchidos com a moda
  Todas as outras colunas estão completas.
""")

    # ──────────────────────────────────────────────────────────────────────────
    # EDA 3.2 — Estatísticas descritivas
    # ──────────────────────────────────────────────────────────────────────────
    print("── 3.2  Estatísticas descritivas ─────────────────────────────────────")
    print("""
POR QUE ESTA ETAPA?
  Média, mediana e desvio-padrão expõem assimetrias e outliers sem precisar
  plotar nada. Fare com mean=32 e max=512 já sinaliza outliers severos.
""")
    print(df.describe().round(2).T)
    print("""
INTERPRETAÇÃO:
  • age   : média 29,7 anos; mediana 28 — distribuição levemente assimétrica
  • fare  : média £32 mas máximo £512 — outliers de primeira classe
  • sibsp : 75% dos passageiros viajavam sem irmãos/cônjuges a bordo
  • parch : 75% viajavam sem pais/filhos
""")

    # ──────────────────────────────────────────────────────────────────────────
    # EDA 3.3 — Distribuição da variável alvo (ANÁLISE DE BALANCEAMENTO)
    # ──────────────────────────────────────────────────────────────────────────
    print("── 3.3  Balanceamento da variável alvo ───────────────────────────────")
    print("""
POR QUE ESTA ETAPA?
  Em classificação binária, classes desbalanceadas fazem com que modelos
  "preguiçosos" prevejam sempre a classe majoritária e ainda obtenham ~62%
  de accuracy. Isso mascara a ausência de aprendizado real.
""")
    vc  = df['survived'].value_counts()
    pct = df['survived'].value_counts(normalize=True) * 100
    print(f"  Não sobreviveu (0): {vc[0]:>4}  ({pct[0]:.1f}%)")
    print(f"  Sobreviveu     (1): {vc[1]:>4}  ({pct[1]:.1f}%)")
    ratio = vc[0] / vc[1]
    print(f"  Razão 0:1 = {ratio:.2f}")

    if ratio > 1.5:
        print("""
DESBALANCEAMENTO DETECTADO (razão > 1,5):
  Com 62% na classe 0, um classificador burro que sempre diz "não sobreviveu"
  teria 62% de accuracy — número enganoso.

  SOLUÇÃO ADOTADA:
    ✓ class_weight='balanced' em todos os modelos sklearn
    ✓ CrossEntropyLoss ponderada no PyTorch
    ✓ F1-score (macro) como métrica principal — penaliza erros nas duas classes
    ✗ SMOTE não aplicado: o dataset é pequeno e SMOTE introduz ruído sintético
      que pode vazar informações no split treino/teste se feito antes da divisão
""")

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].bar(['Não sobreviveu (0)', 'Sobreviveu (1)'], vc.values,
                color=PALETTE, edgecolor='white', linewidth=1.5)
    axes[0].set_title('Distribuição Absoluta — Variável Alvo')
    axes[0].set_ylabel('Quantidade')
    for i, v in enumerate(vc.values):
        axes[0].text(i, v + 5, str(v), ha='center', fontweight='bold')

    axes[1].pie(pct.values, labels=['Não sobreviveu', 'Sobreviveu'],
                colors=PALETTE, autopct='%1.1f%%', startangle=90,
                wedgeprops={'edgecolor': 'white', 'linewidth': 2})
    axes[1].set_title('Proporção das Classes')

    plt.suptitle('Análise de Balanceamento', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(PLOTS / '03_balanceamento.png')
    plt.close()
    print("  → Gráfico salvo: 03_balanceamento.png")

    # ──────────────────────────────────────────────────────────────────────────
    # EDA 3.4 — Sobrevivência por variável categórica
    # ──────────────────────────────────────────────────────────────────────────
    print("\n── 3.4  Sobrevivência por variável categórica ────────────────────────")
    print("""
POR QUE ESTA ETAPA?
  Variáveis categóricas como 'sex' e 'pclass' são frequentemente os
  preditores mais poderosos. Visualizar a taxa de sobrevivência por
  categoria revela quais features têm mais poder discriminativo.
""")
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    for ax, col, title in zip(
        axes,
        ['sex', 'pclass', 'embarked'],
        ['Sexo', 'Classe da Cabine', 'Porto de Embarque']
    ):
        tab = df.groupby(col)['survived'].mean().sort_values(ascending=False)
        bars = ax.bar(tab.index.astype(str), tab.values,
                      color=['#2196F3', '#F44336', '#4CAF50'][:len(tab)],
                      edgecolor='white', linewidth=1.5)
        ax.set_title(f'Taxa de Sobrevivência por {title}')
        ax.set_ylabel('Taxa de sobrevivência')
        ax.set_ylim(0, 1)
        ax.axhline(y=df['survived'].mean(), color='gray', linestyle='--',
                   alpha=0.7, label=f'Média geral ({df["survived"].mean():.2f})')
        ax.legend(fontsize=9)
        for bar in bars:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                    f'{bar.get_height():.2f}', ha='center', fontsize=9)

    plt.suptitle('Taxa de Sobrevivência por Variáveis Categóricas', fontsize=13)
    plt.tight_layout()
    plt.savefig(PLOTS / '03_sobrevivencia_categoricas.png')
    plt.close()

    print("""
INTERPRETAÇÃO:
  • Sexo     : mulheres sobreviveram em 74%, homens apenas 19%
               → feature com maior poder preditivo individual
  • Pclass   : primeira classe 63%, segunda 47%, terceira 24%
               → riqueza/acesso a salva-vidas era decisivo
  • Embarked : Cherbourg (C) 55%, Queenstown (Q) 39%, Southampton (S) 34%
               → correlação com rota/classe, não porto em si
  → Gráfico salvo: 03_sobrevivencia_categoricas.png
""")

    # ──────────────────────────────────────────────────────────────────────────
    # EDA 3.5 — Distribuição de Age e Fare
    # ──────────────────────────────────────────────────────────────────────────
    print("── 3.5  Distribuição de variáveis contínuas ──────────────────────────")
    print("""
POR QUE ESTA ETAPA?
  Histogramas com KDE mostram a forma da distribuição: assimetria, bimodalidade
  e outliers. Features muito assimétricas (como Fare) podem beneficiar-se de
  transformações logarítmicas para melhorar modelos lineares.
""")
    fig, axes = plt.subplots(2, 2, figsize=(13, 9))

    for ax, col in zip(axes[0], ['age', 'fare']):
        sns.histplot(df[col].dropna(), kde=True, ax=ax,
                     color='#2196F3', bins=30, edgecolor='white')
        ax.set_title(f'Distribuição de {col.capitalize()}')
        ax.axvline(df[col].median(), color='red', linestyle='--',
                   label=f'Mediana: {df[col].median():.1f}')
        ax.legend()

    for ax, col in zip(axes[1], ['age', 'fare']):
        sns.boxplot(x='survived', y=col, data=df, ax=ax,
                    palette=PALETTE, width=0.4)
        ax.set_title(f'{col.capitalize()} por Sobrevivência')
        ax.set_xticklabels(['Não sobreviveu', 'Sobreviveu'])

    plt.suptitle('Variáveis Contínuas: Distribuição e Relação com Sobrevivência',
                 fontsize=13)
    plt.tight_layout()
    plt.savefig(PLOTS / '03_variaveis_continuas.png')
    plt.close()

    print("""
INTERPRETAÇÃO:
  • Age  : distribuição bimodal (pico em crianças + adultos jovens)
            sobreviventes ligeiramente mais jovens (mediana ~28 vs ~29)
  • Fare : fortemente assimétrica à direita (outliers de 1ª classe)
            sobreviventes pagaram tarifas MUITO maiores → proxy de classe social
  → Transformação log(Fare+1) será aplicada no pré-processamento
  → Gráfico salvo: 03_variaveis_continuas.png
""")

    # ──────────────────────────────────────────────────────────────────────────
    # EDA 3.6 — Taxa de sobrevivência por faixa etária
    # ──────────────────────────────────────────────────────────────────────────
    print("── 3.6  Sobrevivência por faixa etária ───────────────────────────────")
    df_age = df.dropna(subset=['age']).copy()
    df_age['age_group'] = pd.cut(df_age['age'],
                                  bins=[0, 12, 18, 35, 60, 100],
                                  labels=['Criança\n(0-12)', 'Adolescente\n(13-18)',
                                          'Adulto\n(19-35)', 'Meia-idade\n(36-60)',
                                          'Idoso\n(60+)'])
    taxa = df_age.groupby('age_group', observed=True)['survived'].mean()

    plt.figure(figsize=(10, 5))
    bars = plt.bar(taxa.index.astype(str), taxa.values,
                   color=['#1565C0', '#1976D2', '#42A5F5', '#90CAF9', '#BBDEFB'],
                   edgecolor='white', linewidth=1.5)
    plt.axhline(df['survived'].mean(), color='red', linestyle='--',
                label=f'Média geral ({df["survived"].mean():.2f})', alpha=0.8)
    for bar in bars:
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{bar.get_height():.2f}', ha='center', fontweight='bold')
    plt.title('Taxa de Sobrevivência por Faixa Etária', fontsize=13)
    plt.ylabel('Taxa de sobrevivência'); plt.ylim(0, 1); plt.legend()
    plt.tight_layout()
    plt.savefig(PLOTS / '03_sobrevivencia_idade.png')
    plt.close()

    print("""
INTERPRETAÇÃO:
  • Crianças (0-12) : maior taxa de sobrevivência (~59%) — protocolo "crianças primeiro"
  • Adolescentes    : ~38% — similar à média geral
  • Adultos jovens  : ~37% — maior grupo; muitos homens adultos não sobreviveram
  • Meia-idade      : ~40% — ligeiramente melhor que adultos jovens
  • Idosos          : ~27% — menor mobilidade e acesso aos botes
  → Gráfico salvo: 03_sobrevivencia_idade.png
""")

    # ──────────────────────────────────────────────────────────────────────────
    # EDA 3.7 — Heatmap sexo × classe (combinação de features)
    # ──────────────────────────────────────────────────────────────────────────
    print("── 3.7  Sobrevivência por Sexo × Classe (interação) ──────────────────")
    print("""
POR QUE ESTA ETAPA?
  Features individuais podem esconder interações poderosas. A combinação
  sexo+classe revela padrões que nenhuma feature sozinha captura.
""")
    pivot = df.pivot_table(values='survived', index='sex',
                           columns='pclass', aggfunc='mean')
    plt.figure(figsize=(7, 4))
    sns.heatmap(pivot, annot=True, fmt='.2f', cmap='RdYlGn',
                linewidths=0.5, vmin=0, vmax=1,
                cbar_kws={'label': 'Taxa de sobrevivência'})
    plt.title('Taxa de Sobrevivência: Sexo × Classe da Cabine', fontsize=12)
    plt.xlabel('Classe'); plt.ylabel('Sexo')
    plt.tight_layout()
    plt.savefig(PLOTS / '03_heatmap_sexo_classe.png')
    plt.close()

    print("""
INTERPRETAÇÃO:
  • Mulheres de 1ª classe: 97% de sobrevivência (quase todas salvas)
  • Mulheres de 3ª classe: 50% (acesso restrito aos botes superiores)
  • Homens de 1ª classe: 37% (melhor acesso mas protocolo mulheres/crianças)
  • Homens de 3ª classe: 13% (combinação letal: sexo + classe social)
  → Esta interação sexo×classe é extremamente poderosa para os modelos
  → Gráfico salvo: 03_heatmap_sexo_classe.png
""")

    # ──────────────────────────────────────────────────────────────────────────
    # EDA 3.8 — Correlação com a variável alvo
    # ──────────────────────────────────────────────────────────────────────────
    print("── 3.8  Correlação numérica com 'survived' ───────────────────────────")
    print("""
POR QUE ESTA ETAPA?
  A correlação de Pearson (para numéricas) dá uma triagem rápida de quais
  features têm relação linear com o alvo. Correlações altas (positivas ou
  negativas) indicam features potencialmente úteis para modelos lineares.
""")
    df_enc = df.copy()
    df_enc['sex_num']      = (df_enc['sex'] == 'female').astype(int)
    df_enc['embarked_num'] = df_enc['embarked'].map({'C': 0, 'Q': 1, 'S': 2})
    num_cols = ['survived', 'pclass', 'age', 'sibsp', 'parch',
                'fare', 'sex_num', 'embarked_num']
    corr = df_enc[num_cols].corr()

    plt.figure(figsize=(9, 7))
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
                center=0, linewidths=0.5, square=True)
    plt.title('Matriz de Correlação (features numéricas)', fontsize=12)
    plt.tight_layout()
    plt.savefig(PLOTS / '03_correlacao.png')
    plt.close()

    corr_target = corr['survived'].drop('survived').sort_values(key=abs, ascending=False)
    print("  Correlação com 'survived' (ordem decrescente de valor absoluto):")
    for feat, val in corr_target.items():
        barra = '█' * int(abs(val) * 20)
        sinal = '+' if val > 0 else '-'
        print(f"    {feat:<14}: {val:+.3f}  {sinal}{barra}")

    print("""
INTERPRETAÇÃO:
  • sex_num (+0.54) : maior correlação — ser mulher aumenta chance de sobrevivência
  • pclass  (-0.34) : correlação negativa — classe mais alta → maior sobrevivência
  • fare    (+0.26) : proxy de classe social
  • age     (-0.08) : correlação fraca — relação não-linear com sobrevivência
  → Gráfico salvo: 03_correlacao.png
""")

    # ──────────────────────────────────────────────────────────────────────────
    # EDA 3.9 — Análise de outliers (IQR)
    # ──────────────────────────────────────────────────────────────────────────
    print("── 3.9  Análise de outliers ──────────────────────────────────────────")
    print("""
POR QUE ESTA ETAPA?
  Outliers afetam modelos sensíveis à escala (SVM, Regressão Logística, DL).
  O método IQR identifica valores além de 1,5× o intervalo interquartil.
  Para Fare, outliers são reais (passageiros de luxo) e serão tratados com log.
""")
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, col in zip(axes, ['age', 'fare']):
        sns.boxplot(x=df[col].dropna(), ax=ax,
                    color='#2196F3', width=0.3, flierprops=dict(marker='o',
                    markerfacecolor='red', markersize=5))
        Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
        IQR    = Q3 - Q1
        n_out  = ((df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)).sum()
        ax.set_title(f'{col.capitalize()} — {n_out} outliers (IQR)')
    plt.tight_layout()
    plt.savefig(PLOTS / '03_outliers.png')
    plt.close()

    for col in ['age', 'fare']:
        Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
        IQR    = Q3 - Q1
        n_out  = ((df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)).sum()
        print(f"  {col:<6}: {n_out} outliers  "
              f"(Q1={Q1:.1f}, Q3={Q3:.1f}, IQR={IQR:.1f})")
    print("  → Fare: aplicar log(fare+1) no pré-processamento")
    print("  → Age : outliers são idades reais; não serão removidos")
    print("  → Gráfico salvo: 03_outliers.png")

    # ──────────────────────────────────────────────────────────────────────────
    # EDA 3.10 — Pairplot (relações entre variáveis numéricas)
    # ──────────────────────────────────────────────────────────────────────────
    print("\n── 3.10 Pairplot de variáveis numéricas ──────────────────────────────")
    print("""
POR QUE ESTA ETAPA?
  O pairplot revela relações bivariadas e separa visualmente as classes.
  Pares de features com separação clara entre classes (0 e 1) são candidatos
  fortes para inclusão no modelo.
""")
    pp_cols = ['survived', 'pclass', 'age', 'fare']
    df_pp   = df[pp_cols].dropna().copy()
    g = sns.pairplot(df_pp, hue='survived', palette={0: '#F44336', 1: '#2196F3'},
                     diag_kind='kde', plot_kws={'alpha': 0.4, 's': 20},
                     corner=True)
    g.fig.suptitle('Pairplot — Variáveis Numéricas por Sobrevivência',
                   y=1.02, fontsize=13)
    g.fig.savefig(PLOTS / '03_pairplot.png', bbox_inches='tight')
    plt.close()
    print("""
INTERPRETAÇÃO:
  • pclass × fare : separação clara — 1ª classe paga mais e sobrevive mais
  • age × pclass  : crianças de 1ª e 2ª classe bem separadas
  • age × fare    : dispersão ampla, classes sobrepostas → não-linear
  → Gráfico salvo: 03_pairplot.png
""")

    print("✓  EDA concluída. Todos os gráficos salvos em:", PLOTS)
    

In [8]:
# =============================================================================
# PARTE 4 — PRÉ-PROCESSAMENTO E ENGENHARIA DE FEATURES
# =============================================================================

def preprocess(df: pd.DataFrame):
    """
    Pipeline de pré-processamento com todas as decisões justificadas.

    Etapas:
      A) Descarte de colunas com alta % de nulos ou sem valor preditivo
      B) Imputação de nulos restantes
      C) Engenharia de features (novas variáveis derivadas)
      D) Encoding de categóricas
      E) Normalização de contínuas (StandardScaler)

    Retorna:
        X (ndarray), y (ndarray), feature_names (list), scaler
    """
    print("\n" + "="*70)
    print("PARTE 4 — PRÉ-PROCESSAMENTO")
    print("="*70)

    df = df.copy()

    # A) Descarte de colunas
    # ─────────────────────
    # 'deck'      : 77% nulos — informação insuficiente para imputar
    # 'cabin'     : duplicata de deck em formato raw
    # 'name'      : texto livre; o título (Mr/Mrs) já será extraído
    # 'ticket'    : ID sem padrão semântico claro
    # 'alive'     : duplicata exata de survived (leakage!)
    # 'embark_town': duplicata textual de embarked
    # 'who'/'adult_male': derivadas de sex+age — leakage potencial
    # 'class'     : versão textual de pclass
    drop_cols = ['deck', 'cabin', 'name', 'ticket',
                 'alive', 'embark_town', 'who', 'adult_male', 'class']
    df.drop(columns=[c for c in drop_cols if c in df.columns], inplace=True)
    print(f"  Colunas após descarte: {list(df.columns)}")

    # B) Imputação
    # ────────────
    # age: mediana por pclass (passageiros de 1ª classe eram mais velhos)
    for cls in [1, 2, 3]:
        med = df.loc[df['pclass'] == cls, 'age'].median()
        df.loc[(df['age'].isna()) & (df['pclass'] == cls), 'age'] = med
    # embarked: moda (S = Southampton, maioria)
    df['embarked'].fillna(df['embarked'].mode()[0], inplace=True)
    print(f"  Nulos restantes: {df.isnull().sum().sum()}")

    # C) Engenharia de features
    # ─────────────────────────
    # is_alone   : passageiros sozinhos vs acompanhados — padrão histórico:
    #              famílias grandes tinham dificuldade de se mover
    df['family_size'] = df['sibsp'] + df['parch'] + 1
    df['is_alone']    = (df['family_size'] == 1).astype(int)
    # log_fare   : reduz assimetria e efeito de outliers de 1ª classe
    df['log_fare']    = np.log1p(df['fare'])
    # fare_per_person: tarifas de grupo divididas igualmente
    df['fare_per_person'] = df['fare'] / df['family_size']
    df['log_fare_per_person'] = np.log1p(df['fare_per_person'])
    # age_pclass : interação numérica captura o efeito combinado
    df['age_pclass']  = df['age'] * df['pclass']
    print("  Features derivadas criadas: family_size, is_alone, log_fare, "
          "fare_per_person, log_fare_per_person, age_pclass")

    # D) Encoding de categóricas
    # ──────────────────────────
    df['sex_enc']      = (df['sex'] == 'female').astype(int)  # 1=female
    embarked_dummies   = pd.get_dummies(df['embarked'], prefix='emb', drop_first=True)
    df                 = pd.concat([df, embarked_dummies], axis=1)

    # E) Selecionar features finais
    # ──────────────────────────────
    features = [
        'pclass', 'sex_enc', 'age', 'sibsp', 'parch',
        'log_fare', 'log_fare_per_person',
        'family_size', 'is_alone', 'age_pclass',
        'emb_Q', 'emb_S'
    ]
    # garantir que as dummies existam (caso todos os passageiros sejam C)
    for c in ['emb_Q', 'emb_S']:
        if c not in df.columns:
            df[c] = 0

    X_raw = df[features].values.astype(np.float32)
    y     = df['survived'].values.astype(np.int64)

    # F) Normalização (apenas features contínuas — índices)
    # Binárias (sex_enc, is_alone, emb_*) já estão em {0,1}; normalizá-las
    # distorceria sua semântica e não agrega para tree-based models.
    # Para SVM e DL, normalizamos TUDO (melhora convergência).
    scaler = StandardScaler()
    X      = scaler.fit_transform(X_raw)

    
    print(f"  Shape final: X={X.shape}, y={y.shape}")
    print(f"  Features: {features}")
    return X, y, features, scaler

In [9]:
# =============================================================================
# PARTE 5 — DIVISÃO TREINO / TESTE
# =============================================================================

def split_data(X, y):
    """
    Divisão estratificada 80/20.

    POR QUE ESTRATIFICAR?
    ──────────────────────
    Com 549 instâncias na classe 0 e 342 na classe 1, uma divisão aleatória
    poderia criar splits com proporções bem diferentes. Stratify=y garante
    que ambos os conjuntos mantenham a proporção original ~62/38.
    """
    print("\n" + "="*70)
    print("PARTE 5 — DIVISÃO TREINO / TESTE")
    print("="*70)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, stratify=y, random_state=SEED
    )

    print(f"  Treino : {X_train.shape[0]} amostras  "
          f"(sobrev.: {y_train.sum()}, não-sobrev.: {(y_train==0).sum()})")
    print(f"  Teste  : {X_test.shape[0]} amostras  "
          f"(sobrev.: {y_test.sum()}, não-sobrev.: {(y_test==0).sum()})")
    print(f"  Proporção classe 1 — treino: {y_train.mean():.3f}  "
          f"| teste: {y_test.mean():.3f}  (estratificação OK)")

    return X_train, X_test, y_train, y_test

In [10]:
# =============================================================================
# PARTE 6 — MODELOS TRADICIONAIS (6 MODELOS)
# =============================================================================

def train_traditional_models(X_train, y_train, X_test, y_test):
    """
    Treina 6 classificadores clássicos, cada um com justificativa de escolha
    e explicação dos parâmetros mais relevantes.

    Todos usam class_weight='balanced' quando disponível (tratamento de
    desbalanceamento — Camada 2).
    """
    print("\n" + "="*70)
    print("PARTE 6 — MODELOS TRADICIONAIS")
    print("="*70)

    # ── Descrição de cada modelo ───────────────────────────────────────────────
    descricoes = {
        'LogisticRegression': """
    REGRESSÃO LOGÍSTICA
    ───────────────────
    Modelo linear que estima P(sobreviveu=1|features) usando a função sigmoide.
    É o baseline interpretável: os coeficientes têm significado direto.
    Parâmetros:
      • C=1.0          : inverso da regularização L2 (penaliza coeficientes grandes)
      • max_iter=1000  : iterações do solver 'lbfgs'
      • class_weight='balanced': aumenta peso da classe minoritária
    Quando usar: quando interpretabilidade é prioritária ou como baseline rápido.
""",
        'DecisionTree': """
    ÁRVORE DE DECISÃO
    ─────────────────
    Divide o espaço de features recursivamente por regras if/else. Muito
    interpretável (dá para plotar e explicar a um leigo).
    Parâmetros:
      • max_depth=5    : limita profundidade para evitar overfitting
      • min_samples_leaf=10: nó folha precisa de ≥10 amostras
      • class_weight='balanced'
    Quando usar: para explicar o modelo a stakeholders não-técnicos.
""",
        'RandomForest': """
    RANDOM FOREST
    ─────────────
    Ensemble de 300 árvores treinadas em subsets aleatórios de features e
    amostras (bagging). Reduz variância sem aumentar viés.
    Parâmetros:
      • n_estimators=300: número de árvores (mais = melhor até certo ponto)
      • max_features='sqrt': cada árvore usa √(n_features) features
      • class_weight='balanced'
    Quando usar: quando há features ruidosas e overfitting é uma preocupação.
""",
        'GradientBoosting': """
    GRADIENT BOOSTING
    ─────────────────
    Ensemble sequencial onde cada árvore corrige os erros da anterior (boosting).
    Geralmente o mais poderoso dos métodos tradicionais em dados tabulares.
    Parâmetros:
      • n_estimators=200: número de rounds de boosting
      • learning_rate=0.05: taxa de contribuição de cada árvore
      • max_depth=4: árvores rasas evitam overfitting
      • subsample=0.8: amostragem estocástica por round
    Nota: não aceita class_weight; usa sample_weight passado no .fit().
""",
        'SVM': """
    SVM (Support Vector Machine)
    ─────────────────────────────
    Encontra o hiperplano de margem máxima entre as classes. Com kernel RBF
    é capaz de fronteiras não-lineares no espaço original.
    Parâmetros:
      • C=1.0       : penalidade por violações de margem
      • kernel='rbf': kernel gaussiano → fronteiras não-lineares
      • probability=True: habilita predict_proba (necessário para AUC-ROC)
      • class_weight='balanced'
    Quando usar: datasets pequenos/médios com features bem normalizadas.
""",
        'NaiveBayes': """
    NAIVE BAYES GAUSSIANO
    ─────────────────────
    Assume independência condicional entre features (ingênuo) e distribuições
    Gaussianas. Extremamente rápido e surpreendentemente competitivo em muitos
    problemas reais.
    Parâmetros:
      • var_smoothing=1e-9: suavização numérica da variância
    Nota: não suporta class_weight diretamente; usa priors das classes.
    Quando usar: baseline rápido, dados com muitas features independentes.
""",
    }

    # ── Definição dos modelos ──────────────────────────────────────────────────
    sw = compute_class_weight('balanced', classes=np.array([0, 1]), y=y_train)
    sample_weights = np.where(y_train == 0, sw[0], sw[1])

    modelos = {
        'LogisticRegression': LogisticRegression(
            C=1.0, max_iter=1000, solver='lbfgs',
            class_weight='balanced', random_state=SEED
        ),
        'DecisionTree': DecisionTreeClassifier(
            max_depth=5, min_samples_leaf=10,
            class_weight='balanced', random_state=SEED
        ),
        'RandomForest': RandomForestClassifier(
            n_estimators=300, max_features='sqrt',
            class_weight='balanced', n_jobs=-1, random_state=SEED
        ),
        'GradientBoosting': GradientBoostingClassifier(
            n_estimators=200, learning_rate=0.05,
            max_depth=4, subsample=0.8, random_state=SEED
        ),
        'SVM': SVC(
            C=1.0, kernel='rbf', probability=True,
            class_weight='balanced', random_state=SEED
        ),
        'NaiveBayes': GaussianNB(var_smoothing=1e-9),
    }

    resultados = {}

    for nome, modelo in modelos.items():
        print(descricoes[nome])
        t0 = time.time()

        # GradientBoosting recebe sample_weight no .fit()
        if nome == 'GradientBoosting':
            modelo.fit(X_train, y_train, sample_weight=sample_weights)
        else:
            modelo.fit(X_train, y_train)

        elapsed = time.time() - t0

        y_pred  = modelo.predict(X_test)
        y_proba = modelo.predict_proba(X_test)[:, 1] if hasattr(modelo, 'predict_proba') else None

        acc  = accuracy_score(y_test, y_pred)
        f1   = f1_score(y_test, y_pred, average='macro')
        prec = precision_score(y_test, y_pred, average='macro', zero_division=0)
        rec  = recall_score(y_test, y_pred, average='macro')
        auc  = roc_auc_score(y_test, y_proba) if y_proba is not None else None

        resultados[nome] = {
            'modelo': modelo, 'accuracy': acc, 'f1_macro': f1,
            'precision': prec, 'recall': rec, 'auc_roc': auc,
            'train_time': elapsed, 'y_pred': y_pred, 'y_proba': y_proba
        }
        joblib.dump(modelo, MDIR / f'{nome}.pkl')

        auc_str = f"  AUC={auc:.4f}" if auc else ""
        print(f"    ✓ Acc={acc:.4f}  F1={f1:.4f}  Prec={prec:.4f}  "
              f"Rec={rec:.4f}{auc_str}  ({elapsed:.2f}s)")

    return resultados

In [11]:
# =============================================================================
# PARTE 7 — MODELOS DEEP LEARNING (PyTorch)
# =============================================================================

def _make_loaders(X_train, y_train, X_test, y_test):
    """Converte arrays numpy em DataLoaders PyTorch."""
    def to_t(x, dtype):
        return torch.tensor(x, dtype=dtype, device=DEVICE)

    ds_tr = TensorDataset(to_t(X_train, torch.float32),
                           to_t(y_train, torch.long))
    ds_te = TensorDataset(to_t(X_test,  torch.float32),
                           to_t(y_test,  torch.long))
    return (DataLoader(ds_tr, batch_size=BATCH_SIZE, shuffle=True),
            DataLoader(ds_te, batch_size=BATCH_SIZE, shuffle=False))


def _weighted_criterion(y_train):
    """CrossEntropyLoss com pesos de classe (Camada 4 anti-desbalanceamento)."""
    w = compute_class_weight('balanced', classes=np.array([0,1]), y=y_train)
    return nn.CrossEntropyLoss(
        weight=torch.tensor(w, dtype=torch.float32, device=DEVICE)
    )


def _train_loop(model, loader_tr, loader_te, criterion, optimizer,
                epochs, name, y_train):
    """Loop de treino genérico com early-stop baseado em F1."""
    best_f1, best_state, history = 0.0, None, []
    for ep in range(1, epochs+1):
        model.train()
        ep_loss = 0.0
        for Xb, yb in loader_tr:
            optimizer.zero_grad()
            loss = criterion(model(Xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            ep_loss += loss.item()

        model.eval()
        preds, trues = [], []
        with torch.no_grad():
            for Xb, yb in loader_te:
                preds.extend(model(Xb).argmax(1).cpu().numpy())
                trues.extend(yb.cpu().numpy())

        f1 = f1_score(trues, preds, average='macro', zero_division=0)
        history.append({'epoch': ep, 'loss': ep_loss/len(loader_tr), 'f1': f1})
        if f1 > best_f1:
            best_f1    = f1
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        if ep % 10 == 0:
            print(f"      Época {ep:>3}/{epochs}  loss={ep_loss/len(loader_tr):.4f}  F1={f1:.4f}")

    model.load_state_dict(best_state)
    return model, history, best_f1


# ── Arquitetura 1: MLP ────────────────────────────────────────────────────────
class MLP(nn.Module):
    """
    Multi-Layer Perceptron (Rede Totalmente Conectada)
    ───────────────────────────────────────────────────
    Três camadas ocultas com BatchNorm + ReLU + Dropout.
    BatchNorm estabiliza o treino; Dropout previne overfitting.
    Parâmetros:
      • hidden = (64, 32, 16) : dimensões das camadas ocultas
      • dropout = 0.3         : fração de neurônios desativados aleatoriamente
    """
    def __init__(self, n_features, n_classes=2, hidden=(64, 32, 16), dropout=0.3):
        super().__init__()
        layers, prev = [], n_features
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h),
                       nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, n_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


# ── Arquitetura 2: ResNet tabular ─────────────────────────────────────────────
class ResBlock(nn.Module):
    """Bloco residual: F(x) + x."""
    def __init__(self, dim, dropout=0.2):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(dim, dim), nn.BatchNorm1d(dim), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(dim, dim), nn.BatchNorm1d(dim)
        )
        self.act = nn.ReLU()

    def forward(self, x):
        return self.act(self.block(x) + x)


class TabResNet(nn.Module):
    """
    ResNet Tabular
    ──────────────
    Projeta as features num espaço latente de dimensão `hidden_dim`,
    aplica N blocos residuais e projeta na saída.
    Parâmetros:
      • hidden_dim = 64  : tamanho do espaço latente
      • n_blocks   = 4   : número de blocos residuais
      • dropout    = 0.2
    Por que usar: conexões residuais permitem redes mais profundas sem
    problemas de vanishing gradient.
    """
    def __init__(self, n_features, n_classes=2, hidden_dim=64, n_blocks=4, dropout=0.2):
        super().__init__()
        self.proj   = nn.Sequential(nn.Linear(n_features, hidden_dim),
                                    nn.BatchNorm1d(hidden_dim), nn.ReLU())
        self.blocks = nn.Sequential(*[ResBlock(hidden_dim, dropout) for _ in range(n_blocks)])
        self.out    = nn.Linear(hidden_dim, n_classes)

    def forward(self, x):
        return self.out(self.blocks(self.proj(x)))


# ── Arquitetura 3: Attention MLP ──────────────────────────────────────────────
class AttentionMLP(nn.Module):
    """
    MLP com Self-Attention por Feature
    ────────────────────────────────────
    Aprende a ponderar a importância de cada feature antes do MLP.
    O módulo de atenção produz scores (0-1) por feature; features mais
    relevantes para a predição recebem peso maior.
    Parâmetros:
      • hidden = (64, 32): camadas do MLP após a atenção
    Por que usar: datasets com muitas features irrelevantes se beneficiam
    de mecanismos que "focam" nas features mais discriminativas.
    """
    def __init__(self, n_features, n_classes=2, hidden=(64, 32)):
        super().__init__()
        # Atenção: produz peso por feature
        self.attention = nn.Sequential(
            nn.Linear(n_features, n_features), nn.Tanh(),
            nn.Linear(n_features, n_features), nn.Softmax(dim=1)
        )
        layers, prev = [], n_features
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h),
                       nn.ReLU(), nn.Dropout(0.3)]
            prev = h
        layers.append(nn.Linear(prev, n_classes))
        self.mlp = nn.Sequential(*layers)

    def forward(self, x):
        weights = self.attention(x)  # pesos por feature
        x_w     = x * weights        # features ponderadas
        return self.mlp(x_w)


def train_dl_models(X_train, y_train, X_test, y_test):
    """
    Treina MLP, TabResNet e AttentionMLP com CrossEntropyLoss ponderada.
    """
    print("\n" + "="*70)
    print("PARTE 7 — MODELOS DEEP LEARNING (PyTorch)")
    print("="*70)

    loader_tr, loader_te = _make_loaders(X_train, y_train, X_test, y_test)
    criterion             = _weighted_criterion(y_train)
    n_feat                = X_train.shape[1]

    archs = {
        'MLP':          MLP(n_feat),
        'TabResNet':    TabResNet(n_feat),
        'AttentionMLP': AttentionMLP(n_feat),
    }

    resultados = {}

    for nome, model in archs.items():
        print(f"\n  ── {nome} ──")
        print(f"  {model.__doc__}")
        model.to(DEVICE)
        optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

        t0 = time.time()
        model, history, best_f1 = _train_loop(
            model, loader_tr, loader_te, criterion, optimizer, EPOCHS, nome, y_train
        )
        elapsed = time.time() - t0

        # Avaliação final
        model.eval()
        preds, probas, trues = [], [], []
        with torch.no_grad():
            for Xb, yb in loader_te:
                out = model(Xb)
                probas.extend(torch.softmax(out, 1)[:, 1].cpu().numpy())
                preds.extend(out.argmax(1).cpu().numpy())
                trues.extend(yb.cpu().numpy())

        acc  = accuracy_score(trues, preds)
        f1   = f1_score(trues, preds, average='macro')
        prec = precision_score(trues, preds, average='macro', zero_division=0)
        rec  = recall_score(trues, preds, average='macro')
        auc  = roc_auc_score(trues, probas)

        resultados[nome] = {
            'modelo': model, 'accuracy': acc, 'f1_macro': f1,
            'precision': prec, 'recall': rec, 'auc_roc': auc,
            'train_time': elapsed, 'y_pred': np.array(preds),
            'y_proba': np.array(probas), 'history': history
        }
        torch.save(model.state_dict(), MDIR / f'{nome}.pth')

        # Plot da curva de treino
        ep_list = [h['epoch'] for h in history]
        f1_list = [h['f1']    for h in history]
        plt.figure(figsize=(8, 3))
        plt.plot(ep_list, f1_list, color='#2196F3', linewidth=2)
        plt.axhline(best_f1, color='red', linestyle='--',
                    label=f'Melhor F1={best_f1:.4f}')
        plt.title(f'Curva de Aprendizado — {nome}')
        plt.xlabel('Época'); plt.ylabel('F1-macro (teste)')
        plt.legend(); plt.tight_layout()
        plt.savefig(PLOTS / f'07_learning_curve_{nome}.png')
        plt.close()

        print(f"    ✓ Acc={acc:.4f}  F1={f1:.4f}  Prec={prec:.4f}  "
              f"Rec={rec:.4f}  AUC={auc:.4f}  ({elapsed:.1f}s)")

    return resultados

In [12]:
# =============================================================================
# PARTE 8 — COMPARAÇÃO E ESCOLHA DO MELHOR MODELO
# =============================================================================

def compare_and_select(results_trad, results_dl, y_test):
    """
    Compara todos os modelos e elege o melhor com justificativa detalhada.

    Critério de seleção:
    ─────────────────────
    Score composto = 0.40 × F1-macro + 0.30 × AUC-ROC + 0.30 × Accuracy

    F1-macro (40%): principal métrica em datasets desbalanceados. Penaliza
    modelos que ignoram a classe minoritária.

    AUC-ROC (30%): mede a qualidade do ranking probabilístico — útil para
    calibrar threshold em produção (ex.: alta recall para evacuação).

    Accuracy (30%): importante para comunicação com stakeholders não-técnicos,
    mas subordinada ao F1 para evitar o "paradoxo da acurácia".
    """
    print("\n" + "="*70)
    print("PARTE 8 — COMPARAÇÃO E ESCOLHA DO MELHOR MODELO")
    print("="*70)

    todos = {**results_trad, **results_dl}
    rows  = []
    for nome, r in todos.items():
        rows.append({
            'Modelo':    nome,
            'Accuracy':  r['accuracy'],
            'F1_Macro':  r['f1_macro'],
            'Precision': r['precision'],
            'Recall':    r['recall'],
            'AUC_ROC':   r.get('auc_roc') or 0.0,
            'Tempo(s)':  round(r['train_time'], 2),
            'Tipo':      'DL' if nome in results_dl else 'Trad.',
        })

    df_res = pd.DataFrame(rows)
    df_res['Score'] = (
        0.40 * df_res['F1_Macro'] +
        0.30 * df_res['AUC_ROC']  +
        0.30 * df_res['Accuracy']
    )
    df_res = df_res.sort_values('Score', ascending=False).reset_index(drop=True)

    print("\n  Ranking completo:")
    print("  " + "-"*80)
    print(f"  {'#':<3} {'Modelo':<22} {'F1_Macro':>9} {'AUC_ROC':>8} "
          f"{'Accuracy':>9} {'Score':>7} {'Tipo':<6}")
    print("  " + "-"*80)
    for i, row in df_res.iterrows():
        marker = " ← VENCEDOR" if i == 0 else ""
        print(f"  {i+1:<3} {row['Modelo']:<22} {row['F1_Macro']:>9.4f} "
              f"{row['AUC_ROC']:>8.4f} {row['Accuracy']:>9.4f} "
              f"{row['Score']:>7.4f} {row['Tipo']:<6}{marker}")
    print("  " + "-"*80)

    best_name  = df_res.iloc[0]['Modelo']
    best_result = todos[best_name]

    # Justificativa detalhada
    print(f"""
  JUSTIFICATIVA DA ESCOLHA: {best_name}
  {'─'*50}
  Score composto: {df_res.iloc[0]['Score']:.4f}
    • F1-macro  = {best_result['f1_macro']:.4f}  (peso 40%)
    • AUC-ROC   = {best_result.get('auc_roc', 0):.4f}  (peso 30%)
    • Accuracy  = {best_result['accuracy']:.4f}  (peso 30%)

  Este modelo foi escolhido porque:
    1. Apresenta o melhor equilíbrio entre detectar sobreviventes (recall alto
       na classe 1) e não classificar erroneamente não-sobreviventes como
       sobreviventes (precision alta na classe 1).
    2. O AUC-ROC elevado indica boa capacidade de calibração probabilística,
       importante para ajustar o threshold conforme o contexto de uso.
    3. O F1-macro não é inflado pela classe majoritária, validando que o modelo
       performa bem em ambas as classes.
""")

    # ── Gráfico comparativo ────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    colors = ['#2196F3' if t == 'Trad.' else '#9C27B0'
              for t in df_res['Tipo']]
    bars_col = ['F1_Macro', 'AUC_ROC', 'Score']
    titles   = ['F1-macro', 'AUC-ROC', 'Score Composto']

    for ax, col, title in zip(axes, bars_col, titles):
        vals = df_res[col]
        bars = ax.barh(df_res['Modelo'], vals, color=colors)
        ax.set_title(title, fontweight='bold')
        ax.set_xlim(0.5, 1.0)
        ax.invert_yaxis()
        # Destaque do vencedor
        bars[0].set_edgecolor('gold')
        bars[0].set_linewidth(2.5)

    from matplotlib.patches import Patch
    legend = [Patch(color='#2196F3', label='Tradicional'),
              Patch(color='#9C27B0', label='Deep Learning')]
    fig.legend(handles=legend, loc='lower center', ncol=2, fontsize=10)
    plt.suptitle('Comparação de Modelos — Dataset Titanic', fontsize=13,
                 fontweight='bold')
    plt.tight_layout(rect=[0, 0.05, 1, 1])
    plt.savefig(PLOTS / '08_comparacao_modelos.png', dpi=150)
    plt.close()

    # ── Matrizes de confusão dos Top-3 ────────────────────────────────────────
    top3 = df_res.head(3)['Modelo'].tolist()
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, nome in zip(axes, top3):
        r    = todos[nome]
        cm   = confusion_matrix(y_test, r['y_pred'])
        disp = ConfusionMatrixDisplay(cm, display_labels=['Não sobrev.', 'Sobrev.'])
        disp.plot(ax=ax, colorbar=False, cmap='Blues')
        acc_str = f"Acc={r['accuracy']:.3f}"
        f1_str  = f"F1={r['f1_macro']:.3f}"
        ax.set_title(f'{nome}\n{acc_str}  {f1_str}', fontsize=10)
    plt.suptitle('Matrizes de Confusão — Top 3 Modelos', fontsize=12)
    plt.tight_layout()
    plt.savefig(PLOTS / '08_confusion_top3.png', dpi=150)
    plt.close()

    # ── Curvas ROC dos Top-3 ───────────────────────────────────────────────────
    plt.figure(figsize=(8, 6))
    for nome in top3:
        r = todos[nome]
        if r.get('y_proba') is not None:
            fpr, tpr, _ = roc_curve(y_test, r['y_proba'])
            plt.plot(fpr, tpr, linewidth=2,
                     label=f"{nome} (AUC={r.get('auc_roc', 0):.3f})")
    plt.plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Aleatório')
    plt.xlabel('Taxa de Falso Positivo'); plt.ylabel('Taxa de Verdadeiro Positivo')
    plt.title('Curvas ROC — Top 3 Modelos')
    plt.legend(); plt.tight_layout()
    plt.savefig(PLOTS / '08_roc_curves.png', dpi=150)
    plt.close()

    # Relatório detalhado do melhor
    y_pred = best_result['y_pred']
    print(f"\n  Relatório de Classificação — {best_name}:")
    print("  " + classification_report(y_test, y_pred,
          target_names=['Não sobreviveu', 'Sobreviveu']))

    df_res.to_csv(OUT / '08_comparacao.csv', index=False)
    return best_name, df_res, todos

In [13]:
# =============================================================================
# PARTE 9 — SERIALIZAÇÃO PARA PRODUÇÃO
# =============================================================================

def save_for_production(best_name, todos, scaler, feature_names, y_test):
    """
    Serializa o melhor modelo, o scaler e os metadados para uso em produção.

    Artefatos gerados:
      • production/best_model.*   : modelo serializado (joblib ou .pth)
      • production/scaler.joblib  : StandardScaler para normalização
      • production/config.json    : metadados, features e métricas
      • production/inference.py   : código de inferência pronto para uso
    """
    print("\n" + "="*70)
    print("PARTE 9 — SERIALIZAÇÃO PARA PRODUÇÃO")
    print("="*70)

    r    = todos[best_name]
    modelo = r['modelo']

    # Salvar modelo
    if isinstance(modelo, nn.Module):
        model_path = PROD / 'best_model.pth'
        torch.save(modelo.state_dict(), model_path)
        model_type  = 'pytorch'
        model_class = type(modelo).__name__
    else:
        model_path = PROD / 'best_model.joblib'
        joblib.dump(modelo, model_path)
        model_type  = 'sklearn'
        model_class = type(modelo).__name__

    # Salvar scaler
    scaler_path = PROD / 'scaler.joblib'
    joblib.dump(scaler, scaler_path)

    # Metadados
    config = {
        'best_model_name':  best_name,
        'model_type':       model_type,
        'model_class':      model_class,
        'model_path':       str(model_path),
        'scaler_path':      str(scaler_path),
        'feature_names':    feature_names,
        'n_features':       len(feature_names),
        'n_classes':        2,
        'class_labels':     {0: 'Não sobreviveu', 1: 'Sobreviveu'},
        'threshold':        0.5,
        'imbalance_treatment': {
            'stratified_split':    True,
            'class_weight_balanced': True,
            'metric_f1_macro':     True,
        },
        'metrics_test': {
            'accuracy':  round(r['accuracy'],  4),
            'f1_macro':  round(r['f1_macro'],  4),
            'precision': round(r['precision'], 4),
            'recall':    round(r['recall'],    4),
            'auc_roc':   round(r.get('auc_roc', 0), 4),
        },
        'timestamp': datetime.now().isoformat(),
    }

    config_path = PROD / 'config.json'
    with open(config_path, 'w', encoding='utf-8') as f:
        json.dump(config, f, indent=4, ensure_ascii=False)

    # ── Gerar inference.py ────────────────────────────────────────────────────
    inference_code = f'''# =============================================================================
#  TITANIC — CÓDIGO DE INFERÊNCIA EM PRODUÇÃO
#  Modelo: {best_name}  |  Tipo: {model_type}
#  Gerado em: {datetime.now().strftime("%Y-%m-%d %H:%M")}
# =============================================================================

import json
import numpy as np
import joblib
from pathlib import Path

# Para modelos PyTorch, descomentar:
# import torch
# import torch.nn as nn

# ── Carregamento dos artefatos ─────────────────────────────────────────────────
BASE = Path(__file__).parent
with open(BASE / "config.json") as f:
    CONFIG = json.load(f)

scaler = joblib.load(BASE / "scaler.joblib")

{"# Sklearn:" if model_type == "sklearn" else "# PyTorch:"}
{"model = joblib.load(BASE / 'best_model.joblib')" if model_type == "sklearn" else "# Reconstituir arquitetura e carregar pesos:"}
{"" if model_type == "sklearn" else "# from titanic_pipeline import " + model_class}
{"" if model_type == "sklearn" else "# model = " + model_class + "(n_features=" + str(len(feature_names)) + ")"}
{"" if model_type == "sklearn" else "# model.load_state_dict(torch.load(BASE / 'best_model.pth', map_location='cpu'))"}
{"" if model_type == "sklearn" else "# model.eval()"}

FEATURE_NAMES = {feature_names}


def preprocess_single(pclass, sex, age, sibsp, parch, fare, embarked):
    """
    Pré-processa um único passageiro para inferência.

    Args:
        pclass   : 1, 2 ou 3
        sex      : 'male' ou 'female'
        age      : idade em anos (float)
        sibsp    : nº de irmãos/cônjuge a bordo
        parch    : nº de pais/filhos a bordo
        fare     : tarifa paga (float)
        embarked : 'C', 'Q' ou 'S'

    Returns:
        np.ndarray de shape (1, n_features) normalizado
    """
    family_size        = sibsp + parch + 1
    is_alone           = int(family_size == 1)
    log_fare           = np.log1p(fare)
    fare_per_person    = fare / family_size
    log_fare_per_person = np.log1p(fare_per_person)
    age_pclass         = age * pclass
    sex_enc            = int(sex == "female")
    emb_Q              = int(embarked == "Q")
    emb_S              = int(embarked == "S")

    row = np.array([[
        pclass, sex_enc, age, sibsp, parch,
        log_fare, log_fare_per_person,
        family_size, is_alone, age_pclass,
        emb_Q, emb_S
    ]], dtype=np.float32)

    return scaler.transform(row)


def predict(pclass, sex, age, sibsp, parch, fare, embarked, threshold=0.5):
    """
    Realiza predição para um único passageiro.

    Returns:
        dict com 'classe', 'label', 'probabilidade_sobrevivencia'
    """
    X = preprocess_single(pclass, sex, age, sibsp, parch, fare, embarked)

    {"prob = model.predict_proba(X)[0, 1]" if model_type == "sklearn" else
    "with torch.no_grad():"}
    {"" if model_type == "sklearn" else
    "    out = model(torch.tensor(X, dtype=torch.float32))"}
    {"" if model_type == "sklearn" else
    "    prob = torch.softmax(out, 1)[0, 1].item()"}

    classe = int(prob >= threshold)
    return {{
        "classe":                   classe,
        "label":                    CONFIG["class_labels"][str(classe)],
        "probabilidade_sobrevivencia": round(prob, 4),
        "threshold_usado":          threshold,
    }}


# ── Exemplo de uso ─────────────────────────────────────────────────────────────
if __name__ == "__main__":
    # Rose: mulher, 1ª classe, 17 anos
    resultado = predict(
        pclass=1, sex="female", age=17,
        sibsp=1, parch=2, fare=263.0, embarked="S"
    )
    print("Rose:", resultado)

    # Jack: homem, 3ª classe, 20 anos
    resultado = predict(
        pclass=3, sex="male", age=20,
        sibsp=0, parch=0, fare=7.25, embarked="S"
    )
    print("Jack:", resultado)
'''

    with open(PROD / 'inference.py', 'w', encoding='utf-8') as f:
        f.write(inference_code)

    print(f"""
  Artefatos salvos em {PROD}/:
    ✓ {model_path.name:<30} modelo serializado
    ✓ scaler.joblib                  normalizador
    ✓ config.json                    metadados completos
    ✓ inference.py                   código de inferência pronto

  Para usar em outro arquivo:
    from titanic_output.production.inference import predict
    resultado = predict(pclass=1, sex='female', age=25, ...)
""")

In [14]:
# =============================================================================
# PIPELINE PRINCIPAL
# =============================================================================

def main():
    print("\n" + "█"*70)
    print("  TITANIC — PIPELINE DIDÁTICO DE CLASSIFICAÇÃO")
    print("  Dataset: clássico Titanic (seaborn/Kaggle)")
    print("█"*70)

    # 1. Carregar
    df = load_data()

    # 2. EDA completa
    run_eda(df)

    # 3. Pré-processar
    X, y, feature_names, scaler = preprocess(df)

    # 4. Dividir
    X_train, X_test, y_train, y_test = split_data(X, y)

    # 5. Modelos tradicionais
    results_trad = train_traditional_models(X_train, y_train, X_test, y_test)

    # 6. Modelos DL
    results_dl = train_dl_models(X_train, y_train, X_test, y_test)

    # 7. Comparar e selecionar
    best_name, df_res, todos = compare_and_select(results_trad, results_dl, y_test)

    # 8. Serializar
    save_for_production(best_name, todos, scaler, feature_names, y_test)

    print("\n" + "█"*70)
    print(f"  ✓  Pipeline concluído | Melhor modelo: {best_name}")
    print(f"  ✓  Gráficos  → {PLOTS}")
    print(f"  ✓  Produção  → {PROD}")
    print("█"*70 + "\n")


if __name__ == '__main__':
    main()


██████████████████████████████████████████████████████████████████████
  TITANIC — PIPELINE DIDÁTICO DE CLASSIFICAÇÃO
  Dataset: clássico Titanic (seaborn/Kaggle)
██████████████████████████████████████████████████████████████████████

PARTE 2 — CARREGAMENTO DO DATASET
Dataset carregado:  891 passageiros × 15 colunas
Colunas: ['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked', 'class', 'who', 'adult_male', 'deck', 'embark_town', 'alive', 'alone']

PARTE 3 — ANÁLISE EXPLORATÓRIA DE DADOS (EDA)

── 3.1  Visão geral ──────────────────────────────────────────────────

POR QUE ESTA ETAPA?
  Antes de qualquer análise precisamos entender o que temos: quantas linhas,
  quais tipos de dado e se há valores ausentes. Colunas com muitos nulos
  precisam de estratégia especial (imputação ou descarte).

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       ----------